# The Metric That Divides by Almost Zero

**Topic:** Regression metrics &nbsp;·&nbsp; **Difficulty:** Easy

MAPE is popular with stakeholders because a percentage is easy to explain. It is
also undefined at zero, unbounded near zero, and asymmetric: under-predicting is
capped at 100% error for non-negative forecasts while over-predicting is not
(`pred_a` goes negative on some quiet days, so its under-predictions can exceed
100% too). On demand data with quiet days
in it, one near-zero actual can dominate the whole metric.

Compute four metrics on the same predictions and watch them disagree about which
model is better.

## The data (created in the next cell)

- `y_true` `(1000,)`: daily demand, including some near-zero days.
- `pred_a`: a model that is good on average but occasionally far off.
- `pred_b`: a model that over-predicts by 6% of demand plus 1.5 units.

## Your task

With **NumPy only**:

1. Implement MAE, RMSE, MAPE (`mean(|y - p| / |y|)`, skipping exact zeros), and
   sMAPE (`mean(2|y - p| / (|y| + |p|))`).
2. Score both models with all four.
3. `winner_by`: a length-4 array holding `0.0` if model A wins that metric and
   `1.0` if model B does, in the order `[MAE, RMSE, MAPE, sMAPE]`.
4. Set `answers` to a dict with `"metrics_a"`, `"metrics_b"` (length-4 each, same
   order), `"winner_by"`, and `"worst_mape_contrib"`, the largest single-point
   contribution to model A's MAPE, as a fraction of the total.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the metrics do not all pick the same winner.

In [ ]:
import numpy as np

rng = np.random.default_rng(3014)
n = 1000
y_true = np.abs(rng.gamma(2.0, 30.0, n))
quiet = rng.choice(n, 25, replace=False)
y_true[quiet] = rng.uniform(0.05, 1.5, 25)          # near-zero days

pred_a = y_true + rng.normal(0, 4.0, n)
pred_b = y_true * 1.06 + 1.5
print(f"y_true {y_true.shape}, min = {y_true.min():.3f}, median = {np.median(y_true):.1f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the data created above to compute the metrics, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("metrics_a", [4], 7.82365784385806, 0.1343176844634737, 1e-06),
    ("metrics_b", [4], 10.942750068958553, 0.17223382305472934, 1e-06),
    ("winner_by", [4], 2.0, 0.04570247555718865, 1e-06),
    ("worst_mape_contrib", [], 0.06276088979792098, -0.010508558795755442, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "winner_by" in answers:
    _w = np.asarray(answers["winner_by"], dtype=float)
    _split = len(set(_w.tolist())) > 1
    print(f"  {'metrics disagree':<24} {'ok' if _split else 'wrong'}  "
          f"(winners {[int(v) for v in _w]} for MAE, RMSE, MAPE, sMAPE)")
    _ok = _ok and _split

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

On a day when actual demand is 0.05 and you predict 8, the absolute error is
under 8 units but the percentage error is 15,900%, and MAPE averages that in with
everything else, so a handful of quiet days decide the ranking. sMAPE bounds each
term at 2 and behaves far better, though it is still asymmetric. The general rule:
never use a relative metric when the denominator can approach zero, and always
report an absolute metric alongside it.

```python
import numpy as np

def mae(y, p):
    return float(np.mean(np.abs(y - p)))

def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))

def mape(y, p):
    m = y != 0
    return float(np.mean(np.abs(y[m] - p[m]) / np.abs(y[m])))

def smape(y, p):
    return float(np.mean(2 * np.abs(y - p) / (np.abs(y) + np.abs(p))))

metrics_a = np.array([mae(y_true, pred_a), rmse(y_true, pred_a),
                      mape(y_true, pred_a), smape(y_true, pred_a)])
metrics_b = np.array([mae(y_true, pred_b), rmse(y_true, pred_b),
                      mape(y_true, pred_b), smape(y_true, pred_b)])

winner_by = (metrics_b < metrics_a).astype(float)

contrib = np.abs(y_true - pred_a) / np.abs(y_true)
worst = float(contrib.max() / contrib.sum())

answers = {
    "metrics_a": metrics_a,
    "metrics_b": metrics_b,
    "winner_by": winner_by,
    "worst_mape_contrib": worst,
}
```